# 日频 AlphaMining：Colab A100 一体化训练流水线

本 Notebook 在一次 Colab 会话中依次完成：数据预处理 → 表达式检查 → GFlowNet 训练 → AlphaEval 筛选 → LightGBM 融合 → 训练产物打包下载。

RQAlphaPlus **不在 Colab 运行**。下载本 Notebook 生成的 `alphamining_colab_outputs.zip` 后，在本地使用 `06_rqalpha_backtest.ipynb` 回测。

## 1. 克隆仓库

默认克隆主仓库。需要使用其他 fork 时，可预先设置环境变量 `ALPHAMINING_REPO_URL`。

In [ ]:
from pathlib import Path
import os

if not Path('configs/training_config.yaml').exists():
    repo = os.environ.get(
        'ALPHAMINING_REPO_URL',
        'https://github.com/JacksonChiy/AlphaMining-GFlowNet-AlphaEval.git',
    )
    !git clone $repo
    %cd AlphaMining-GFlowNet-AlphaEval

## 2. 安装依赖

In [ ]:
%pip install -q -r requirements.txt

## 3. 检查 A100、CUDA 与显存

正式训练必须通过 A100 校验。

In [ ]:
import torch

print('PyTorch 版本:', torch.__version__)
print('CUDA 状态:', torch.cuda.is_available())
print('CUDA 运行时:', torch.version.cuda)
assert torch.cuda.is_available(), '请在 Colab 中启用 NVIDIA A100 GPU 运行时'
gpu = torch.cuda.get_device_properties(0)
print('GPU 型号:', gpu.name)
print('GPU 显存（GB）:', round(gpu.total_memory / 1024**3, 2))
assert 'A100' in gpu.name.upper(), f'正式训练要求 A100；当前检测到 {gpu.name}'

## 4. 加载配置

`FAST_MODE=True` 默认读取快速训练配置：使用 2024 年以来的数据、以前置半年成交额选择 800 只股票，并生成 20 个因子。确认流程可运行后，改为 `False` 即切换到全量正式配置。

In [ ]:
import yaml

FAST_MODE = True
CONFIG_PATH = ('configs/quick_training_config.yaml' if FAST_MODE
               else 'configs/training_config.yaml')
DOWNLOAD_OUTPUTS = True

with open(CONFIG_PATH, encoding='utf-8') as file:
    config = yaml.safe_load(file)
POOL_SIZE = int(config.get('pipeline', {}).get('pool_size', 100))
print('训练模式:', '快速' if FAST_MODE else '正式全量')
print('配置:', CONFIG_PATH, '因子池:', POOL_SIZE)
config

## 5. 从 Google Drive 复制 `price.csv`

把行情文件放在 Google Drive 的 `MyDrive/price.csv`。挂载 Drive 后，代码按文件大小判断是否需要复制到 Colab 本地高速磁盘。代码不会调用外部行情接口。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil

PRICE_CSV_DRIVE = Path('/content/drive/MyDrive/price.csv')
PRICE_CSV_LOCAL = Path('data/price.csv')
PRICE_CSV_LOCAL.parent.mkdir(parents=True, exist_ok=True)
if not PRICE_CSV_DRIVE.exists():
    raise FileNotFoundError(f'Google Drive 中没有找到: {PRICE_CSV_DRIVE}')
if (not PRICE_CSV_LOCAL.exists() or
        PRICE_CSV_LOCAL.stat().st_size != PRICE_CSV_DRIVE.stat().st_size):
    print('复制price.csv到Colab本地，请等待...')
    shutil.copy2(PRICE_CSV_DRIVE, PRICE_CSV_LOCAL)
config['dataset']['file'] = str(PRICE_CSV_LOCAL)
print('local csv MB =', round(PRICE_CSV_LOCAL.stat().st_size / 1024**2, 1))

## 6. 数据预处理

自动完成字段映射、日期转换、排序、缺失与异常处理、可选复权及同日截面标准化，生成 `data/daily_price.pkl`。

In [ ]:
import json
from src.data_loader import prepare_price_csv

dataset_filter_keys = (
    'start_date', 'end_date', 'max_stocks', 'universe_start_date',
    'universe_end_date', 'chunksize',
)
dataset_filters = {
    key: config['dataset'][key]
    for key in dataset_filter_keys
    if config['dataset'].get(key) is not None
}
daily = prepare_price_csv(
    config['dataset']['file'],
    config['dataset']['output'],
    'results/data_quality_report.json',
    **dataset_filters,
)
display(daily.head())
print('行数:', len(daily), '交易日数:', daily.date.nunique(), '股票数:', daily.code.nunique())
display(json.loads(Path('results/data_quality_report.json').read_text(encoding='utf-8')))

## 7. 表达式引擎检查

随机生成并执行 10 个表达式，确认复杂度、深度和有效值覆盖率。

In [ ]:
import pandas as pd
from src.expression import Expression

expression_rows = []
for seed in range(10):
    expression = Expression.generate(max_depth=4, seed=seed)
    values = expression.execute(daily)
    expression_rows.append({
        'expression': str(expression),
        'complexity': expression.complexity(),
        'depth': expression.depth(),
        'coverage': round(values.notna().mean(), 4),
    })
display(pd.DataFrame(expression_rows))

## 8. GFlowNet A100 混合精度训练

训练使用 Trajectory Balance Loss，开启 A100 混合精度和 TF32，并对同一 epoch 的轨迹执行批量 Transformer 推理；Reward 按配置使用多线程并行，RankIC、LongIR 与风险暴露使用向量化计算。覆盖率同时检查有效观测与有效交易日占比，低覆盖表达式会被降权，低于配置门槛的表达式不会进入因子池。每条训练轨迹仍会输出完整进度，但 GPU 指标按 epoch 批量传回 CPU，减少同步等待。

In [ ]:
from src.gflownet.run_training import run as run_gflownet

experiment_dir = run_gflownet(CONFIG_PATH, require_a100=True, pool_size=POOL_SIZE)
print('实验目录:', experiment_dir.resolve())
display(pd.read_csv('results/gflownet_training_metrics.csv').tail())
display(pd.read_csv('results/gflownet_trajectory_metrics.csv').tail(32))
display(pd.read_csv('results/alpha_pool.csv').head(20))

## 9. 检查点重载测试

In [ ]:
from src.gflownet import GFlowNetTrainer, RewardEvaluator

reward_evaluator = RewardEvaluator(daily, **config['reward'])
loaded_trainer = GFlowNetTrainer.load_checkpoint(
    'checkpoints/gflownet_best.pt', reward_evaluator, device='cuda'
)
sample_expression, _, sample_tokens = loaded_trainer.sample_trajectory(greedy=True)
print('检查点重载成功:', sample_expression)
print('Token 序列:', sample_tokens)
assert Path('results/alpha_pool.csv').exists()
assert Path('results/alpha_factor_matrix.pkl').exists()

## 10. AlphaEval 因子评价与 DPP 筛选

标签统一为 `close(t+5) / close(t+1) - 1`。

In [ ]:
import shutil
from src.alpha_eval import AlphaEval, AlphaEvalConfig

factor_matrix = pd.read_pickle('results/alpha_factor_matrix.pkl')
factor_metadata = pd.read_csv('results/alpha_pool.csv')
alpha_eval_values = dict(config['alpha_eval'])
alpha_eval_values['horizon'] = config['dataset']['horizon']
alpha_eval = AlphaEval(daily, factor_matrix, AlphaEvalConfig(**alpha_eval_values))
evaluation = alpha_eval.evaluate(factor_metadata)
selected_factors = evaluation.loc[evaluation['dpp_selected'].astype(bool), 'factor'].tolist()
shutil.copy2('results/alpha_eval_result.csv', experiment_dir / 'alpha_eval_result.csv')
print('DPP 选中因子数:', len(selected_factors))
display(evaluation.head(30))

## 11. LightGBM 滚动融合

使用 purge 滚动训练，生成本地 RQAlphaPlus 回测所需的 `prediction_score.csv`。

In [ ]:
from src.model import LightGBMConfig, LightGBMFusion

fusion = LightGBMFusion(LightGBMConfig(**config['lightgbm']))
prediction = fusion.fit_predict(
    daily, factor_matrix, selected_factors, output_dir='results/lightgbm'
)
shutil.copy2('results/lightgbm/model_metrics.csv', experiment_dir / 'lgbm_model_metrics.csv')
prediction.to_csv(experiment_dir / 'prediction_score.csv', index=False)
loaded_lgbm = LightGBMFusion.load('results/lightgbm/lgbm_model.joblib')
print('LightGBM 模型重载成功；特征数:', len(loaded_lgbm['features']))
display(prediction.tail(30))

## 12. 打包并下载 Colab 训练产物

压缩包包含配置、GFlowNet 检查点、因子池、因子矩阵、AlphaEval 结果、LightGBM 模型与预测分数。下载后在本地仓库根目录解压，再运行 `notebooks/06_rqalpha_backtest.ipynb`。

In [ ]:
from google.colab import files
from zipfile import ZIP_DEFLATED, ZipFile

archive_entries = [
    (Path(CONFIG_PATH), 'results/colab_training_config.yaml'),
    (Path('checkpoints/gflownet_best.pt'), 'checkpoints/gflownet_best.pt'),
    (Path('results/data_quality_report.json'), 'results/data_quality_report.json'),
    (Path('results/gflownet_training_metrics.csv'), 'results/gflownet_training_metrics.csv'),
    (Path('results/gflownet_trajectory_metrics.csv'), 'results/gflownet_trajectory_metrics.csv'),
    (Path('results/alpha_pool.csv'), 'results/alpha_pool.csv'),
    (Path('results/alpha_factor_matrix.pkl'), 'results/alpha_factor_matrix.pkl'),
    (Path('results/alpha_eval_result.csv'), 'results/alpha_eval_result.csv'),
    (Path('results/lightgbm/lgbm_model.joblib'), 'results/lightgbm/lgbm_model.joblib'),
    (Path('results/lightgbm/model_metrics.csv'), 'results/lightgbm/model_metrics.csv'),
    (Path('results/lightgbm/feature_importance.csv'), 'results/lightgbm/feature_importance.csv'),
    (Path('results/lightgbm/prediction_score.csv'), 'results/lightgbm/prediction_score.csv'),
]
missing_outputs = [str(path) for path, _ in archive_entries if not path.exists()]
if missing_outputs:
    raise FileNotFoundError(f'缺少训练产物: {missing_outputs}')

archive_path = Path('/content/alphamining_colab_outputs.zip')
with ZipFile(archive_path, 'w', ZIP_DEFLATED) as archive:
    for path, archive_name in archive_entries:
        archive.write(path, arcname=archive_name)
print('产物压缩包:', archive_path, '大小（MB）:', round(archive_path.stat().st_size / 1024**2, 2))
print('本地回测入口: notebooks/06_rqalpha_backtest.ipynb')
if DOWNLOAD_OUTPUTS:
    files.download(str(archive_path))